In [1]:
import os
import sys
import copy

import torch

sys.path.append('../../../')

%load_ext autoreload
%autoreload 2
    
from computer_vision.yolov11_pose.nn.tasks import PoseModel

In [2]:
cfg='../yolo11-pose.yaml'
model=PoseModel(cfg=cfg,nc=1,verbose=True)
model.fuse(); # we need to call `fuse` so we can successfully load pretrained weight

In nn.tasks.DetectionModel.__init__ input 1 is not equal to nc in yaml 80->overwrite yaml by input
In nn.tasks.parse_model nc 1, act None, scales {'n': [0.5, 0.25, 1024], 's': [0.5, 0.5, 1024], 'm': [0.5, 1.0, 512], 'l': [1.0, 1.0, 512], 'x': [1.0, 1.5, 512]}
In nn.tasks.parse_model depth 1.0, width 1.0, kpt_shape [17, 3]
In nn.tasks.parse_model no model scale passed. Assuming scale=n.
In nn.tasks.parse_model depth 0.5, width 0.25, max_channels 1024

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  Conv                                         [3, 16, 3, 2]                 
  1                  -1  1      4672  Conv                                         [16, 32, 3, 2]                
  2                  -1  1      6640  C3k2                                         [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  Conv                                         [64, 6

In [3]:
model_dirpath='D:/results/yolov11_pose/predict'
checkpoint_file=os.path.join(model_dirpath, 'yolo11n_torch.pt')
assert os.path.isfile(checkpoint_file), f'{checkpoint_file} does not exist'
checkpoint=torch.load(checkpoint_file, weights_only=False)
try:
    model.load_state_dict(checkpoint['model'])
except RuntimeError as err:
    state_dict=copy.deepcopy(model.state_dict())
    for name, params in checkpoint['model'].items():
        name=name[len('model.'):] # each parameter name is model.model.xxx so we need to remove 1 model.
        
        if name not in state_dict or params.shape!=state_dict[name].shape:
            print(name, name in state_dict, (params.shape,state_dict[name].shape) if name in state_dict else None)
        else: state_dict[name]=params
    model.load_state_dict(state_dict)

In [ ]:
from computer_vision.yolov11.data.utils import imread
from computer_vision.yolov11.data.augment import LetterBox # for preprocessing
from computer_vision.yolov11.utils.nms import non_max_suppression # for post processing
from computer_vision.yolov11.utils.ops import scale_boxes # for post processing, i.e, scaling boxes back to original image size